In [5]:
cd github/brave-childcare-godot/

/Users/brylew/github/brave-childcare-godot


In [65]:
import pandas as pd

persons_df = pd.read_json("Sample Inputs/persons_nopoison.json")
persons_df.head()


,pid,start_poison,type
0,1507817,0,generic
1,1293512,0,generic
2,4397533,0,generic
3,1752380,0,generic
4,3436464,0,generic


### Define the population in our scenario
6 infants (under 12 months) (group1).   
8 younger toddlers (12-24 months) (group2).   
10 older toddlers or “twos” (24-36 months) (group3).   
16 preschoolers (3-5 years) (group4).   
Total providers: 2+2+2+2 = 8 +2 floaters   


In [66]:
### Define population
group_names = ["infants", "younger toddlers","older toddlers","preschoolers","providers", "floaters"]
group_sizes = [6, 8, 10, 16, 8, 2]


In [67]:
# Build group counts from cell 3
if len(group_names) != len(group_sizes):
    raise ValueError("group_names and group_sizes must have the same length.")

group_counts = dict(zip(group_names, group_sizes))

rows = []

if len(group_names) > 9:
    raise ValueError("This pid scheme supports at most 9 groups (first digit 1-9).")

for group_idx, group_name in enumerate(group_names, start=1):
    count = group_counts[group_name]
    if count > 99:
        raise ValueError(f"Group '{group_name}' has {count} rows, but max is 99 for 3-digit pid format.")

    for i in range(1, count + 1):
        pid = f"{group_idx}{i:02d}"  # e.g., 101, 102, ..., 201, 202, ...
        rows.append({
            "pid": pid,
            "start_poison": 0,
            "type": "generic",
            "role": group_name,
        })

group_persons_df = pd.DataFrame(rows)
group_persons_df.tail()


,pid,start_poison,type,role
45,506,0,generic,providers
46,507,0,generic,providers
47,508,0,generic,providers
48,601,0,generic,floaters
49,602,0,generic,floaters


In [68]:
group_persons_df.role.value_counts()

preschoolers        16
older toddlers      10
younger toddlers     8
providers            8
infants              6
floaters             2
Name: role, dtype: int64

In [69]:
output_path = "Sample Inputs/persons_childcare_pop.json"

group_persons_df.to_json(output_path, orient="records")
print(f"Saved {len(group_persons_df)} rows to {output_path}")


Saved 50 rows to Sample Inputs/persons_childcare_pop.json


In [70]:
example_schedule_entry = {
    "day": 0,
    "pid": "211964",
    "activity_number": 1,
    "aid": "1",
    "start_time": 24332,
    "duration": 856,
    "oid": "/root/Main/Map/Level1/EntranceContainer/Entrance1"
  }


In [71]:
sched_template = pd.read_csv("analysis/day_schedule.csv")

In [74]:
sched_template

,Time,Group1 (Infants),Group2 (Toddlers),Group3 (Twos),Group4 (Preschool)
0,6:30-8:00,Arrival/Freeplay,Arrival/Freeplay,Arrival/Freeplay,Arrival/Freeplay
1,8:00-8:30,Diapers,Breakfast,Breakfast,Breakfast
2,8:30-9:00,Feeding,Diapers,Diapers/Potty,Potty
3,9:00-9:30,Morning nap,Morning circle,Morning circle,Morning circle
4,9:30-10:00,Morning nap,Small groups,Small groups,Guided centers
5,10:00-10:30,Tummy time,Outside play,Outside play,Outside play
6,10:30-11:00,Cribs & Swings,Story time,Story time,Story time
7,11:00-11:30,Diapers,Lunch,Lunch,Lunch
8,11:30-12:00,Feeding,Diapers/Potty/Lullabies,Diapers/Potty/Lullabies,Potty
9,12:00-2:30,Nap time,Nap time,Nap time,Nap time


In [73]:
objects_df = pd.read_json("outputs/childcare_objects_mar20.json", orient="records")
objects_df.sample(10)

,group,oid,pos_x,pos_y,type
61,K_1_1,/root/Main/Map/Cafeteria1/CafeteriaTable9,-1032,2727,cafeteria_table
26,CA_1_8,/root/Main/Map/Level1/CubicleContainer/Cubicle27,285,-90,cubicle
4,CA_1_7,/root/Main/Map/Level1/CubicleContainer/Cubicle16,-865,1103,cubicle
32,CR_1_1280,/root/Main/Map/Level1/ConferenceTableContainer...,-1275,-35,conference_table
58,K_1_1,/root/Main/Map/Cafeteria1/CafeteriaTable12,-752,2497,cafeteria_table
52,G_1,/root/Main/Map/Gym1/GymEquipment3,816,2583,gym_equipment
19,CA_1_8,/root/Main/Map/Level1/CubicleContainer/Cubicle20,100,638,cubicle
1,CA_1_7,/root/Main/Map/Level1/CubicleContainer/Cubicle13,-1304,995,cubicle
41,WC_1_3,/root/Main/Map/Level1/RestroomContainer/Restroom6,-173,1039,restroom
60,K_1_1,/root/Main/Map/Cafeteria1/CafeteriaTable8,-791,2726,cafeteria_table


In [75]:
# Build role -> activity -> [oid options] from the schedule template
sched_template_clean = sched_template.copy()
sched_template_clean.columns = [c.replace("\xa0", "").strip() for c in sched_template_clean.columns]

role_col = "role" if "role" in group_persons_df.columns else "type"

# Map person roles to schedule-template columns
column_by_role = {
    "infants": next(c for c in sched_template_clean.columns if c.startswith("Group1")),
    "younger toddlers": next(c for c in sched_template_clean.columns if c.startswith("Group2")),
    "older toddlers": next(c for c in sched_template_clean.columns if c.startswith("Group3")),
    "preschoolers": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
    "providers": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
    "floaters": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
}

# Collect concrete oid instances from objects_df
valid_objects = objects_df.dropna(subset=["type", "oid"]).copy()
valid_objects["type"] = valid_objects["type"].astype(str).str.strip()
valid_objects["oid"] = valid_objects["oid"].astype(str).str.strip()
valid_objects = valid_objects[valid_objects["oid"].str.startswith("/")]

oids_by_type = (
    valid_objects.sort_values("oid")
    .groupby("type", sort=False)["oid"]
    .apply(list)
    .to_dict()
)

fallback_oid = next(iter(oids_by_type.get("cubicle", [])), None)
if fallback_oid is None:
    raise ValueError("objects_df must include at least one cubicle oid.")

activity_to_object_type = {
    "Arrival/Freeplay": "entrance",
    "Freeplay/Departure": "entrance",
    "Outside play": "gym_equipment",
    "Breakfast": "cafeteria_table",
    "Lunch": "cafeteria_table",
    "Feeding": "cafeteria_table",
    "Potty/Snack": "cafeteria_table",
    "Diapers": "restroom",
    "Potty": "restroom",
    "Diapers/Potty": "restroom",
    "Diapers/Potty/Lullabies": "restroom",
    "Diapers/Potty/Snack": "restroom",
    "Potty/Music": "restroom",
    "Diapers/Music": "restroom",
    "Morning nap": "cubicle",
    "Nap time": "cubicle",
    "Cribs & Swings": "cubicle",
    "Tummy time": "cubicle",
    "Morning circle": "conference_table",
    "Afternoon circle": "conference_table",
    "Story time": "conference_table",
    "Small groups": "cubicle",
    "Guided centers": "cubicle",
    "Arts & Crafts": "conference_table",
}

# Optional role preferences when selecting object instances
# Tuned for realistic zone segregation and supervision patterns
role_type_preferences = {
    "infants": {
        "entrance": ["Entrance1", "Entrance2"],  # Primary infant entrances
        "cubicle": ["Cubicle1", "Cubicle2", "Cubicle3", "Cubicle4", "Cubicle5"],  # Dedicated infant sleep zone
        "restroom": ["Restroom1", "Restroom2"],  # Infant diaper change stations
        "cafeteria_table": ["CafeteriaTable1", "CafeteriaTable2"],  # Infant feeding areas
    },
    "younger toddlers": {
        "entrance": ["Entrance2", "Entrance1"],  # Shared with infants
        "cubicle": ["Cubicle6", "Cubicle7", "Cubicle8", "Cubicle9", "Cubicle10"],  # Toddler sleep zone
        "restroom": ["Restroom3", "Restroom4"],  # Toddler potty training areas
        "cafeteria_table": ["CafeteriaTable3", "CafeteriaTable4", "CafeteriaTable5"],
        "conference_table": ["ConferenceTable1", "ConferenceTable2"],
    },
    "older toddlers": {
        "entrance": ["Entrance2", "Entrance3"],  # More independent
        "cubicle": ["Cubicle11", "Cubicle12", "Cubicle13", "Cubicle14", "Cubicle15"],  # Older toddler sleep
        "restroom": ["Restroom4", "Restroom5", "Restroom6"],  # Potty training areas
        "cafeteria_table": ["CafeteriaTable6", "CafeteriaTable7", "CafeteriaTable8"],
        "conference_table": ["ConferenceTable2", "ConferenceTable3"],
        "gym_equipment": ["GymEquipment1", "GymEquipment2"],  # Limited active play
    },
    "preschoolers": {
        "entrance": ["Entrance3", "Entrance1"],  # Main entrance
        "cubicle": ["Cubicle16", "Cubicle17", "Cubicle18", "Cubicle19", "Cubicle20"],  # Preschool sleep
        "restroom": ["Restroom6", "Restroom7"],  # Independent bathrooms
        "cafeteria_table": ["CafeteriaTable9", "CafeteriaTable10", "CafeteriaTable11"],
        "conference_table": ["ConferenceTable4", "ConferenceTable5", "ConferenceTable6", "ConferenceTable7"],  # More group activities
        "gym_equipment": ["GymEquipment2", "GymEquipment3", "GymEquipment4"],  # Active play areas
    },
    "providers": {
        "entrance": ["Entrance1", "Entrance2", "Entrance3"],  # Full access for supervision
        "cubicle": ["Cubicle1", "Cubicle11", "Cubicle16"],  # Can be in any zone
        "restroom": ["Restroom1", "Restroom4", "Restroom7"],  # Distributed for support
        "cafeteria_table": ["CafeteriaTable1", "CafeteriaTable6", "CafeteriaTable10"],  # Scattered
        "conference_table": ["ConferenceTable3", "ConferenceTable5"],  # For activities
        "gym_equipment": ["GymEquipment2"],  # Can supervise active play
    },
    "floaters": {
        "entrance": ["Entrance1", "Entrance2"],  # More restricted to supervision areas
        "cubicle": ["Cubicle5", "Cubicle10", "Cubicle15"],  # Can move between zones
        "restroom": ["Restroom2", "Restroom5"],  # Support in key areas
        "cafeteria_table": ["CafeteriaTable3", "CafeteriaTable8", "CafeteriaTable11"],  # Can help with meals
        "conference_table": ["ConferenceTable2", "ConferenceTable6"],  # Activity support
        "gym_equipment": ["GymEquipment1", "GymEquipment3"],  # Supervise play
    },
}


def ranked_oids_for_role(role_name, object_type, max_options=3):
    candidates = list(oids_by_type.get(object_type, []))
    if not candidates:
        return [fallback_oid]

    prefs = role_type_preferences.get(role_name, {}).get(object_type, [])
    ranked = []

    for hint in prefs:
        ranked.extend([oid for oid in candidates if hint in oid and oid not in ranked])

    ranked.extend([oid for oid in candidates if oid not in ranked])
    return ranked[:max_options]


role_activity_oid_options = {}
for role_name in sorted(group_persons_df[role_col].dropna().astype(str).unique()):
    sched_col = column_by_role.get(role_name, next(c for c in sched_template_clean.columns if c.startswith("Group4")))

    activities = sorted(
        {
            str(a).strip()
            for a in sched_template_clean[sched_col].dropna()
            if str(a).strip()
        }
    )

    role_activity_oid_options[role_name] = {}
    for activity in activities:
        object_type = activity_to_object_type.get(activity, "cubicle")
        role_activity_oid_options[role_name][activity] = ranked_oids_for_role(role_name, object_type, max_options=3)

# Backward-compatible single-default map (first option)
group_activity_oid_map = {
    role_name: {activity: options[0] for activity, options in activity_map.items()}
    for role_name, activity_map in role_activity_oid_options.items()
}

# Preview: each role has multiple candidate oids for each activity
{role: {a: oids[:2] for a, oids in list(activity_map.items())[:3]} for role, activity_map in role_activity_oid_options.items()}


{'floaters': {'Arrival/Freeplay': ['/root/Main/Map/Level1/EntranceContainer/Entrance1',
   '/root/Main/Map/Cafeteria1/Entrance'],
  'Arts & Crafts': ['/root/Main/Map/Level1/ConferenceTableContainer/ConferenceTable2',
   '/root/Main/Map/Level1/ConferenceTableContainer/ConferenceTable6'],
  'Breakfast': ['/root/Main/Map/Cafeteria1/CafeteriaTable3',
   '/root/Main/Map/Cafeteria1/CafeteriaTable8']},
 'infants': {'Arrival/Freeplay': ['/root/Main/Map/Level1/EntranceContainer/Entrance1',
   '/root/Main/Map/Cafeteria1/Entrance'],
  'Cribs & Swings': ['/root/Main/Map/Level1/CubicleContainer/Cubicle1',
   '/root/Main/Map/Level1/CubicleContainer/Cubicle10'],
  'Diapers': ['/root/Main/Map/Level1/RestroomContainer/Restroom1',
   '/root/Main/Map/Level1/RestroomContainer/Restroom2']},
 'older toddlers': {'Afternoon circle': ['/root/Main/Map/Level1/ConferenceTableContainer/ConferenceTable2',
   '/root/Main/Map/Level1/ConferenceTableContainer/ConferenceTable3'],
  'Arrival/Freeplay': ['/root/Main/Map/C

In [76]:
import random


def _clock_to_minutes(clock_text):
    h, m = clock_text.split(":")
    return int(h) * 60 + int(m)


def _normalize_time_window(start_txt, end_txt, prev_start_min):
    start_min = _clock_to_minutes(start_txt)
    end_min = _clock_to_minutes(end_txt)

    if prev_start_min is not None:
        while start_min < prev_start_min:
            start_min += 12 * 60

    while end_min <= start_min:
        end_min += 12 * 60

    return start_min, end_min


sched_template_clean = sched_template.copy()
sched_template_clean.columns = [c.replace("\xa0", "").strip() for c in sched_template_clean.columns]

role_col = "role" if "role" in group_persons_df.columns else "type"

role_schedule_col = {
    "infants": next(c for c in sched_template_clean.columns if c.startswith("Group1")),
    "younger toddlers": next(c for c in sched_template_clean.columns if c.startswith("Group2")),
    "older toddlers": next(c for c in sched_template_clean.columns if c.startswith("Group3")),
    "preschoolers": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
    "providers": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
    "floaters": next(c for c in sched_template_clean.columns if c.startswith("Group4")),
}

max_start_jitter_sec = 600
rng = random.Random(42)

schedule_rows = []

for _, person in group_persons_df.iterrows():
    person_pid = str(person["pid"])
    person_role = str(person[role_col])
    sched_col = role_schedule_col.get(person_role, next(c for c in sched_template_clean.columns if c.startswith("Group4")))

    activity_number = 1
    prev_start_min = None

    for _, slot in sched_template_clean.iterrows():
        time_range = str(slot["Time"]).strip()
        if not time_range or "-" not in time_range:
            continue

        start_txt, end_txt = [t.strip() for t in time_range.split("-", 1)]
        start_min, end_min = _normalize_time_window(start_txt, end_txt, prev_start_min)
        prev_start_min = start_min

        activity_name = str(slot[sched_col]).strip()
        if not activity_name:
            continue

        oid_options = role_activity_oid_options.get(person_role, {}).get(activity_name, [fallback_oid])

        pid_num = int(person_pid) if person_pid.isdigit() else sum(ord(ch) for ch in person_pid)
        option_idx = (pid_num + activity_number) % len(oid_options)
        oid = oid_options[option_idx]

        base_start_sec = start_min * 60
        base_duration_sec = (end_min - start_min) * 60

        start_jitter_sec = rng.randint(0, max_start_jitter_sec)
        jittered_start_sec = base_start_sec + start_jitter_sec
        jittered_duration_sec = max(1, base_duration_sec - start_jitter_sec)

        schedule_rows.append({
            "day": 0,
            "pid": person_pid,
            "activity_number": activity_number,
            "aid": str(activity_number),
            "start_time": jittered_start_sec,
            "duration": jittered_duration_sec,
            "oid": oid,
        })

        activity_number += 1

full_schedule_df = pd.DataFrame(schedule_rows)
full_schedule_df.head()


,day,pid,activity_number,aid,start_time,duration,oid
0,0,101,1,1,23514,5286,/root/Main/Map/Level1/EntranceContainer/Entrance1
1,0,101,2,2,28825,1775,/root/Main/Map/Level1/RestroomContainer/Restroom2
2,0,101,3,3,30881,1519,/root/Main/Map/Cafeteria1/CafeteriaTable11
3,0,101,4,4,32650,1550,/root/Main/Map/Level1/CubicleContainer/Cubicle1
4,0,101,5,5,34428,1572,/root/Main/Map/Level1/CubicleContainer/Cubicle10


In [28]:
full_schedule_df.pid.value_counts()

101    15
414    15
404    15
405    15
406    15
407    15
408    15
409    15
410    15
411    15
412    15
413    15
415    15
102    15
416    15
501    15
502    15
503    15
504    15
505    15
506    15
507    15
508    15
601    15
403    15
402    15
401    15
310    15
103    15
104    15
105    15
106    15
201    15
202    15
203    15
204    15
205    15
206    15
207    15
208    15
301    15
302    15
303    15
304    15
305    15
306    15
307    15
308    15
309    15
602    15
Name: pid, dtype: int64

In [77]:
schedule_output_path = "Sample Inputs/schedule_grouped_childcare.json"

full_schedule_df.to_json(schedule_output_path, orient="records", indent=2)
print(f"Saved {len(full_schedule_df)} rows to {schedule_output_path}")


Saved 750 rows to Sample Inputs/schedule_grouped_childcare.json


In [44]:
full_schedule_df.oid.unique()

array(['/root/Main/Map/Level1/EntranceContainer/Entrance1',
       '/root/Main/Map/Level1/RestroomContainer/Restroom1',
       '/root/Main/Map/Cafeteria1/CafeteriaTable1',
       '/root/Main/Map/Level1/CubicleContainer/Cubicle1',
       '/root/Main/Map/Level1/ConferenceTableContainer/ConferenceTable1',
       '/root/Main/Map/Gym1/GymEquipment1'], dtype=object)

In [45]:
sched_set = set(full_schedule_df.oid.unique())
object_set = set(objects_df["oid"].dropna().unique())
print(f"OIDs in schedule but not in objects_df: {sched_set - object_set}")

OIDs in schedule but not in objects_df: set()


In [49]:
sched_set = set(full_schedule_df.pid.unique())
person_set = set(group_persons_df["pid"].dropna().unique())
print(f"PIDs in schedule but not in persons_df: {sched_set - person_set}")

PIDs in schedule but not in persons_df: set()


In [50]:
group_persons_df

,pid,start_poison,type
0,101,0,infants
1,102,0,infants
2,103,0,infants
3,104,0,infants
4,105,0,infants
5,106,0,infants
6,201,0,younger toddlers
7,202,0,younger toddlers
8,203,0,younger toddlers
9,204,0,younger toddlers


In [57]:
# Sanity checks for role-based oid assignment
sched_set = set(full_schedule_df["oid"].dropna().unique())
object_set = set(objects_df["oid"].dropna().unique())
print(f"OIDs in schedule but not in objects_df: {sched_set - object_set}")

# Show a few role/activity pools and how many options each has
sample_role = "infants" if "infants" in role_activity_oid_options else next(iter(role_activity_oid_options))
print(f"Sample role: {sample_role}")
print({k: len(v) for k, v in list(role_activity_oid_options[sample_role].items())[:6]})


OIDs in schedule but not in objects_df: set()
Sample role: infants
{'Arrival/Freeplay': 3, 'Cribs & Swings': 3, 'Diapers': 3, 'Diapers/Music': 3, 'Feeding': 3, 'Freeplay/Departure': 3}


In [79]:
# Summary: Role-based location segregation with tuned preferences
print("\n" + "="*80)
print("ZONE SEGREGATION SUMMARY - TUNED ROLE PREFERENCES")
print("="*80)

# Extract instance names from OIDs
full_schedule_df['instance'] = full_schedule_df['oid'].str.extract(r'/([\w\d]+)$')[0]

# Add role to schedule for analysis
full_schedule_df_with_role = full_schedule_df.merge(
    group_persons_df[['pid', role_col]].astype({'pid': str}),
    left_on='pid', right_on='pid', how='left'
)

# Show location instance distribution by role
print("\nLocation instances used per role:")
for role in sorted(group_persons_df[role_col].unique()):
    role_schedule = full_schedule_df_with_role[full_schedule_df_with_role[role_col] == role]
    instances = sorted(role_schedule['instance'].unique())
    print(f"  {role:20s}: {instances}")



ZONE SEGREGATION SUMMARY - TUNED ROLE PREFERENCES

Location instances used per role:
  floaters            : ['CafeteriaTable3', 'CafeteriaTable8', 'ConferenceTable1', 'ConferenceTable2', 'ConferenceTable6', 'Cubicle10', 'Cubicle15', 'Cubicle5', 'Entrance', 'Entrance1', 'GymEquipment1', 'GymEquipment2', 'GymEquipment3', 'Restroom1', 'Restroom2', 'Restroom5']
  infants             : ['CafeteriaTable1', 'CafeteriaTable10', 'CafeteriaTable11', 'Cubicle1', 'Cubicle10', 'Cubicle11', 'Entrance', 'Entrance1', 'Restroom1', 'Restroom2', 'Restroom3']
  older toddlers      : ['CafeteriaTable6', 'CafeteriaTable7', 'CafeteriaTable8', 'ConferenceTable1', 'ConferenceTable2', 'ConferenceTable3', 'Cubicle11', 'Cubicle12', 'Cubicle13', 'Entrance', 'Entrance1', 'GymEquipment1', 'GymEquipment2', 'GymEquipment3', 'Restroom4', 'Restroom5', 'Restroom6']
  preschoolers        : ['CafeteriaTable10', 'CafeteriaTable11', 'CafeteriaTable9', 'ConferenceTable4', 'ConferenceTable5', 'ConferenceTable6', 'Cubicle16',